# 상품 분류 파인튜닝 — 최종 점검

튜닝은 끝났다. 이 노트북은 **결론 재확인 + 남은 두 가지 점검**만 한다 — 기존 세션에
안 이어도 된다(전부 디스크에 저장된 산출물만 읽는다). `../CLASSIFIER_FINETUNE_BRIEF.md`,
`aihub_pipeline_server.ipynb`(전체 파이프라인 기록)와 같이 참고.

In [11]:
# --- 환경 설정 ---
import json
import subprocess
import sys
import threading
import time
from pathlib import Path


def run(script, *args, heartbeat_every=30):
    """스크립트를 실시간 스트리밍으로 실행한다. GPU 작업이 죽었는지 살았는지
    구분하려고 heartbeat를 같이 찍는다."""
    cmd = [sys.executable, str(HERE / script), *[str(a) for a in args]]
    print("$ " + " ".join(cmd) + "\n")
    start = time.time()
    process = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", bufsize=1,
    )
    stop_heartbeat = threading.Event()

    def heartbeat():
        while not stop_heartbeat.wait(heartbeat_every):
            print(f"  ... 아직 실행 중 (경과 {int(time.time() - start)}초)")

    thread = threading.Thread(target=heartbeat, daemon=True)
    thread.start()
    for line in process.stdout:
        print(f"[{time.time() - start:7.1f}s] {line}", end="")
    process.wait()
    stop_heartbeat.set()
    elapsed = time.time() - start
    status = "성공" if process.returncode == 0 else f"⚠ 실패(returncode={process.returncode})"
    print(f"\n{status} — 총 {elapsed:.1f}초")
    return process


HERE = Path("/home/j-i15a108/work/S15P11A108/AI/validation/CLASS/product-info-finetune")
assert (HERE / "aihub_dataset.py").exists(), f"HERE가 틀렸다: {HERE}"
ARTIFACTS = HERE / "artifacts"
VARIANT = "masked"
print("good")

good


## 1. 지금까지 결론

- **카테고리**: last-N=4(비전 인코더 마지막 4블록 언프리즈) — test **0.8424**
- **색상**: LoRA r=16·alpha=32, 카테고리 체크포인트에서 이어받은 순차학습 — test **0.7770**
- 백본 하나 공유(순차학습된 backbone으로 양쪽 다 서빙)는 폐기 — sklearn 프로브로 확인한
  결과 색상-적응 backbone은 카테고리에서 zero-shot(0.6763)보다도 낮았다
  (last-N 0.6495, LoRA 0.6217). **두 모델을 따로 배포한다.**

In [12]:
run(
    "aihub_finetune_summary.py",
    "--artifacts-finetune-dir", HERE / "artifacts_finetune",
    "--best-per-group", "method,task",
    "--sort-by", "test_accuracy",
    "--out", HERE / "artifacts_finetune" / "leaderboard_final.md",
)

$ /home/j-i15a108/work/.venv-aihub/bin/python /home/j-i15a108/work/S15P11A108/AI/validation/CLASS/product-info-finetune/aihub_finetune_summary.py --artifacts-finetune-dir /home/j-i15a108/work/S15P11A108/AI/validation/CLASS/product-info-finetune/artifacts_finetune --best-per-group method,task --sort-by test_accuracy --out /home/j-i15a108/work/S15P11A108/AI/validation/CLASS/product-info-finetune/artifacts_finetune/leaderboard_final.md

[    0.0s] 저장: /home/j-i15a108/work/S15P11A108/AI/validation/CLASS/product-info-finetune/artifacts_finetune/leaderboard_final.md
[    0.0s] | run | method | task | variant | best_epoch | best_val_accuracy | test_accuracy | batch_size | dropped_test_rows | dropped_val_rows | epochs | init_checkpoint | lora_alpha | lora_dropout | lora_r | lr_adapter | lr_backbone | lr_head | model_id | target_modules | total_params | trainable_params | unfreeze_layers |
[    0.0s] | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | ---

<Popen: returncode: 0 args: ['/home/j-i15a108/work/.venv-aihub/bin/python', ...>

## 2. 남은 점검 ① — 미커버 24종(신발·가방 등) zero-shot이 last-N=4 백본에서도 유지되는가

우리 정답셋(`product-info-check/inputs/` + `data/ground_truth.json`)에서 last-N=4 헤드가
모르는 클래스(=미커버 클래스)만 골라, 원본(동결) backbone과 last-N=4 backbone 각각으로
zero-shot 정확도를 재서 비교한다. 색상 순차학습이 카테고리를 zero-shot 이하로 깎아먹는 걸
이미 확인했으니(`aihub_finetune_probe.py`), 카테고리 학습도 미커버 클래스를 깎아먹었을
가능성을 같은 방식으로 확인하는 것이다.

In [13]:
run(
    "aihub_ground_truth_zero_shot_check.py",
    "--ground-truth-dir", HERE.parent / "product-info-check",
    "--checkpoint", HERE / "artifacts_finetune" / "vit_last4_category_masked_lr1e-05" / "checkpoint.pt",
    "--method", "last-n",
    "--unfreeze-layers", 4,
)

$ /home/j-i15a108/work/.venv-aihub/bin/python /home/j-i15a108/work/S15P11A108/AI/validation/CLASS/product-info-finetune/aihub_ground_truth_zero_shot_check.py --ground-truth-dir /home/j-i15a108/work/S15P11A108/AI/validation/CLASS/product-info-check --checkpoint /home/j-i15a108/work/S15P11A108/AI/validation/CLASS/product-info-finetune/artifacts_finetune/vit_last4_category_masked_lr1e-05/checkpoint.pt --method last-n --unfreeze-layers 4

[    8.5s] 미커버 클래스 24종 중 정답셋에서 찾은 이미지: 5장
[    8.5s] 대상 클래스: ['COTTON_PANTS', 'CROSS_BAG', 'DRESS_SHOES', 'HOOD_ZIPUP']
[    8.5s] 
[    8.5s] 원본 백본(동결)  : 4/5 = 0.8000
[    8.5s] 파인튜닝된 백본  : 4/5 = 0.8000
[    8.5s] 
[    8.5s] 원본 대비 예측이 달라진 이미지:
[    8.5s]   🟢 greyp.png (정답=COTTON_PANTS): 원본=CARGO_PANTS → 파인튜닝=COTTON_PANTS
[    8.5s]   🔴 hzip.png (정답=HOOD_ZIPUP): 원본=HOOD_ZIPUP → 파인튜닝=JACKET

성공 — 총 9.1초


<Popen: returncode: 0 args: ['/home/j-i15a108/work/.venv-aihub/bin/python', ...>

## 3. 남은 점검 ② — 지연시간 실측

last-N=4·LoRA-웜 각각 forward 시간(랜덤 텐서 기준)과, 두 모델 순차/스레드 동시 실행 시
합산 시간을 잰다. `--sample-image`로 실제 사진 한 장을 주면 PIL 디코드+전처리까지 포함한
진짜 end-to-end 숫자(원본/last-N/LoRA 개별 + 디코드 1회 공유하는 두 모델 합산)도 같이
낸다 — backbone forward만으로는 실제 병목(이미지 디코드)이 빠진다.

In [15]:
run(
    "aihub_finetune_latency.py",
    "--last-n-checkpoint", HERE / "artifacts_finetune" / "vit_last4_category_masked_lr1e-05" / "checkpoint.pt",
    "--unfreeze-layers", 4,
    "--lora-checkpoint", HERE / "artifacts_finetune" / "lora_r16_a32_d0.05_qv_color_masked_lr0.0001_warm" / "checkpoint.pt",
    "--r", 16, "--alpha", 32,
    "--sample-image", HERE.parent / "product-info-check" / "inputs" / "apc.png",
)

$ /home/j-i15a108/work/.venv-aihub/bin/python /home/j-i15a108/work/S15P11A108/AI/validation/CLASS/product-info-finetune/aihub_finetune_latency.py --last-n-checkpoint /home/j-i15a108/work/S15P11A108/AI/validation/CLASS/product-info-finetune/artifacts_finetune/vit_last4_category_masked_lr1e-05/checkpoint.pt --unfreeze-layers 4 --lora-checkpoint /home/j-i15a108/work/S15P11A108/AI/validation/CLASS/product-info-finetune/artifacts_finetune/lora_r16_a32_d0.05_qv_color_masked_lr0.0001_warm/checkpoint.pt --r 16 --alpha 32 --sample-image /home/j-i15a108/work/S15P11A108/AI/validation/CLASS/product-info-check/inputs/apc.png

[   14.2s] device=cpu warmup=3 iterations=20
[   14.2s] 
[   14.2s] 원본(zero-shot) backbone       : p50=14.1ms p95=14.5ms
[   14.2s] last-N=4 backbone             : p50=15.5ms p95=27.7ms
[   14.2s] LoRA r=16 backbone               : p50=17.0ms p95=22.9ms
[   14.2s] 
[   14.2s] 순차 실행(두 모델 합산)         : p50=31.5ms p95=48.9ms
[   14.2s] 스레드 동시 실행(CPU 병렬 시도)  : p50=77.3ms p95=145.9

<Popen: returncode: 0 args: ['/home/j-i15a108/work/.venv-aihub/bin/python', ...>

## 4. 산출물 확인

In [16]:
for path in sorted(ARTIFACTS.glob("*")):
    size = path.stat().st_size if path.is_file() else 0
    print(f"{size / 1e6:10.2f} MB  {path.name}")

for meta_file in sorted((HERE / "artifacts_finetune").glob("*/meta.json")):
    print("\n--- " + str(meta_file.relative_to(HERE)) + " ---")
    print(json.dumps(json.loads(meta_file.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))

      0.00 MB  crop_preview
      0.00 MB  eval_report.md
      0.00 MB  eval_report_color.md
      0.02 MB  head_category_masked.joblib
      0.03 MB  head_color_masked.joblib
      0.03 MB  head_color_masked_clothes.joblib
      0.03 MB  head_color_masked_clothes_v2.joblib
      0.03 MB  head_color_raw_clothes.joblib
      0.00 MB  head_polo_masked.joblib
     28.82 MB  index.jsonl
     16.11 MB  sample.jsonl
      0.02 MB  scan.txt
      0.64 MB  test_masked.jsonl
      0.00 MB  test_masked.meta.json
      2.51 MB  test_masked.npy
      0.64 MB  test_masked_pixels.jsonl
      0.00 MB  test_masked_pixels.meta.json
    368.49 MB  test_masked_pixels.npy
      0.64 MB  test_raw.jsonl
      0.00 MB  test_raw.meta.json
      2.51 MB  test_raw.npy
     13.89 MB  train_masked.jsonl
      0.00 MB  train_masked.meta.json
     54.16 MB  train_masked.npy
     13.89 MB  train_masked_pixels.jsonl
      0.00 MB  train_masked_pixels.meta.json
   7961.43 MB  train_masked_pixels.npy
     13.90 MB  tr